# 11 — Dataset com censura + modelo de tempo até conclusão

## Objetivo

A base atual tem contratos concluídos **e** contratos ainda em andamento. Por isso, a regressão direta de `target_dias_atraso` deixa de ser o alvo principal: ela só funciona quando a conclusão já é conhecida.

Este notebook adota uma abordagem mais adequada ao cenário atual:

- mantém todos os contratos selecionados;
- trata contratos ainda não concluídos como **censurados à direita**;
- cria um dataset com **uma linha por contrato**;
- usa como target de sobrevivência o par:
  - `duracao_observada_dias`
  - `evento_conclusao`
- mantém `target_atraso_conhecido` apenas como diagnóstico auxiliar;
- compara modelos AFT Log-Normal regulares em validação temporal;
- escolhe automaticamente o melhor candidato pelo **Harrell C-index médio**;
- calcula, para contratos em andamento, a probabilidade condicional de ultrapassar o prazo vigente.

### Por que este target é mais apropriado?

Para um contrato concluído, sabemos o tempo real até a conclusão.

Para um contrato ainda em andamento, não sabemos quanto ele ainda vai durar, mas sabemos que **até a data de corte ele ainda não terminou**. Isso é exatamente um problema de censura, e não um valor ausente comum.

Assim, em vez de descartar os contratos em andamento ou inventar um atraso final, usamos a informação parcial de forma estatisticamente correta.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import brier_score_loss

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

# Data de corte do conjunto analisado.
# Atualize esta data quando os arquivos forem atualizados.
DATA_CORTE_ANALISE = pd.Timestamp("2026-10-14")

OUTPUT_DIR = Path("/mnt/data/11_survival_aft_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data de corte:", DATA_CORTE_ANALISE.date())
print("Saídas:", OUTPUT_DIR)

## 1. Localização e carga dos arquivos

In [ ]:
def localizar(arquivos):
    for nome in arquivos:
        p = Path("/mnt/data") / nome
        if p.exists():
            return p
    raise FileNotFoundError(
        "Nenhum dos arquivos foi encontrado: "
        + ", ".join(arquivos)
    )

PATH_CONTRATOS = localizar([
    "contratos_com_conclusao_andamento - Página1.csv",
])

PATH_PARCELAS = localizar([
    "vw_bi_parcelas(1).csv",
    "vw_bi_parcelas.csv",
])

PATH_ADITIVOS_PRAZO = localizar([
    "vw_taditivoprazo(1).csv",
    "vw_taditivoprazo.csv",
])

PATH_ADITIVOS_VALOR = localizar([
    "vw_taditivovalor(1).csv",
    "vw_taditivovalor.csv",
])

PATH_LIBERACOES = localizar([
    "vw_tparcela_liberacao(1).csv",
    "vw_tparcela_liberacao.csv",
])

contratos = pd.read_csv(
    PATH_CONTRATOS,
    encoding="utf-8-sig",
    low_memory=False,
)

parcelas = pd.read_csv(
    PATH_PARCELAS,
    encoding="utf-8-sig",
    low_memory=False,
)

aditivos_prazo = pd.read_csv(
    PATH_ADITIVOS_PRAZO,
    encoding="utf-8-sig",
    low_memory=False,
)

aditivos_valor = pd.read_csv(
    PATH_ADITIVOS_VALOR,
    encoding="utf-8-sig",
    low_memory=False,
)

liberacoes = pd.read_csv(
    PATH_LIBERACOES,
    encoding="utf-8-sig",
    low_memory=False,
)

print("Contratos:", contratos.shape)
print("Parcelas:", parcelas.shape)
print("Aditivos de prazo:", aditivos_prazo.shape)
print("Aditivos de valor:", aditivos_valor.shape)
print("Liberações:", liberacoes.shape)

## 2. Tratamento de datas e filtro para os contratos selecionados

In [ ]:
# Datas da tabela mestre
for coluna in [
    "data_os",
    "data_termino_original",
    "data_termino_vigente_final",
    "data_conclusao_definitiva",
]:
    contratos[coluna] = pd.to_datetime(
        contratos[coluna],
        errors="coerce",
    )

# Datas de aditivos de prazo
for coluna in [
    "Data_termino_anterior",
    "Data_termino_nova",
    "Data_assinatura_ta_prazo",
]:
    aditivos_prazo[coluna] = pd.to_datetime(
        aditivos_prazo[coluna],
        errors="coerce",
    )

# Aditivos de valor
aditivos_valor["Data_assinatura_TA_valor"] = pd.to_datetime(
    aditivos_valor["Data_assinatura_TA_valor"],
    errors="coerce",
)

# Parcelas
for coluna in [
    "data_prevista_parcela",
    "data_efetiva",
    "data_liberacao",
]:
    parcelas[coluna] = pd.to_datetime(
        parcelas[coluna],
        errors="coerce",
    )

# Liberações
for coluna in [
    "Data_efet_conclusao",
    "Data_liberação",
]:
    liberacoes[coluna] = pd.to_datetime(
        liberacoes[coluna],
        errors="coerce",
    )

ids = set(contratos["id_contrato"])

parcelas = parcelas[
    parcelas["id_contrato"].isin(ids)
].copy()

aditivos_prazo = aditivos_prazo[
    aditivos_prazo["idContrato"].isin(ids)
].copy()

aditivos_valor = aditivos_valor[
    aditivos_valor["idContrato"].isin(ids)
].copy()

liberacoes = liberacoes[
    liberacoes["id_Contrato"].isin(ids)
].copy()

assert contratos["id_contrato"].is_unique

print("Contratos selecionados:", len(contratos))
print()
print(contratos["status_final_controle"].value_counts(dropna=False))

## 3. Reconstrução do prazo vigente

Para contratos concluídos, o prazo vigente é reconstruído usando o último aditivo de prazo assinado **até a conclusão**.

Para contratos ainda não concluídos, usa-se o último aditivo assinado **até a data de corte**.

Se não houver aditivo de prazo, vale o prazo original:

`data_os + prazo_original_dias`

In [ ]:
df = contratos.copy()

df["data_termino_original_calc"] = (
    df["data_os"]
    + pd.to_timedelta(
        df["prazo_original_dias"],
        unit="D",
    )
)

# Um contrato é considerado evento concluído quando possui conclusão
# definitiva conhecida até a data de corte, independentemente do texto
# do status cadastral.
df["evento_conclusao"] = (
    df["data_conclusao_definitiva"].notna()
    & (
        df["data_conclusao_definitiva"]
        <= DATA_CORTE_ANALISE
    )
).astype(int)

df["data_fim_observacao"] = (
    df["data_conclusao_definitiva"]
    .where(
        df["evento_conclusao"].eq(1),
        DATA_CORTE_ANALISE,
    )
)


def prazo_vigente_na_observacao(row):
    cutoff = row["data_fim_observacao"]

    ad = aditivos_prazo[
        aditivos_prazo["idContrato"].eq(
            row["id_contrato"]
        )
        & aditivos_prazo[
            "Data_assinatura_ta_prazo"
        ].notna()
        & (
            aditivos_prazo[
                "Data_assinatura_ta_prazo"
            ]
            <= cutoff
        )
        & aditivos_prazo[
            "Data_termino_nova"
        ].notna()
    ].sort_values(
        [
            "Data_assinatura_ta_prazo",
            "Data_termino_nova",
        ]
    )

    if len(ad):
        return ad.iloc[-1]["Data_termino_nova"]

    return row["data_termino_original_calc"]


df["data_termino_vigente_observada"] = (
    df.apply(
        prazo_vigente_na_observacao,
        axis=1,
    )
)

# Validação da reconstrução para os contratos onde a base mestre já
# possui prazo vigente final.
check = df[
    df["data_termino_vigente_final"].notna()
].copy()

check["diferenca_dias"] = (
    check["data_termino_vigente_observada"]
    - check["data_termino_vigente_final"]
).dt.days

print(
    "Contratos com prazo final informado:",
    len(check),
)

print(
    "Reconstruções exatamente iguais:",
    int(check["diferenca_dias"].eq(0).sum()),
)

display(
    check.loc[
        check["diferenca_dias"].ne(0),
        [
            "id_contrato",
            "data_termino_vigente_final",
            "data_termino_vigente_observada",
            "diferenca_dias",
        ],
    ]
)

## 4. Target principal: tempo até conclusão com censura

O target deixa de ser somente `target_dias_atraso`.

Agora usamos duas colunas:

- `duracao_observada_dias`: tempo entre OS e conclusão, ou entre OS e a data de corte quando ainda não concluiu;
- `evento_conclusao`: 1 quando a conclusão é observada e 0 quando o contrato está censurado.

Esse par permite usar os contratos ainda em andamento sem inventar um valor final.

In [ ]:
df["duracao_observada_dias"] = (
    df["data_fim_observacao"]
    - df["data_os"]
).dt.days.clip(lower=1)

df["prazo_vigente_observado_dias"] = (
    df["data_termino_vigente_observada"]
    - df["data_os"]
).dt.days.clip(lower=1)

df["dias_restantes_prazo_na_corte"] = (
    df["data_termino_vigente_observada"]
    - DATA_CORTE_ANALISE
).dt.days

# Target auxiliar de atraso.
# 1 = atraso já conhecido
# 0 = concluiu dentro do prazo
# NaN = ainda está em andamento e ainda não chegou ao prazo
df["target_atraso_conhecido"] = np.nan

concluidos = df["evento_conclusao"].eq(1)

df.loc[
    concluidos,
    "target_atraso_conhecido",
] = (
    df.loc[
        concluidos,
        "data_conclusao_definitiva",
    ]
    > df.loc[
        concluidos,
        "data_termino_vigente_observada",
    ]
).astype(int)

em_andamento_ja_atrasados = (
    df["evento_conclusao"].eq(0)
    & (
        DATA_CORTE_ANALISE
        > df["data_termino_vigente_observada"]
    )
)

df.loc[
    em_andamento_ja_atrasados,
    "target_atraso_conhecido",
] = 1

df["status_modelagem"] = np.where(
    df["evento_conclusao"].eq(1),
    "evento_observado",
    "censurado",
)

print("Total de contratos:", len(df))
print("Conclusões observadas:", int(df["evento_conclusao"].sum()))
print("Censurados:", int(df["evento_conclusao"].eq(0).sum()))
print()
print(
    "Target atraso conhecido:",
    int(df["target_atraso_conhecido"].notna().sum()),
)
print(
    "Atrasos conhecidos:",
    int(df["target_atraso_conhecido"].fillna(0).sum()),
)
print(
    "Atraso ainda desconhecido:",
    int(df["target_atraso_conhecido"].isna().sum()),
)

## 5. Features seguras para o modelo atual

Como o modelo é de tempo até conclusão, usamos apenas informações que podem ser tratadas como características basais do contrato:

- `prazo_original_dias`;
- `classificacao_contrato`;
- quantidade de parcelas do tipo `Serviços Originais`.

Não usamos, neste primeiro modelo, indicadores acumulados ao final da execução dos contratos concluídos. Fazer isso misturaria informação posterior ao início da obra e criaria risco de leakage.

As informações de execução são mantidas no dataset apenas para diagnóstico e para futuros modelos temporais.

In [ ]:
# Quantidade de parcelas originais.
parcelas_originais = (
    parcelas[
        parcelas["tipo_de_parcela"].eq(
            "Serviços Originais"
        )
    ]
    .groupby("id_contrato")
    .agg(
        qtd_parcelas_originais=(
            "id_parcela",
            "nunique",
        )
    )
)

df = df.merge(
    parcelas_originais,
    left_on="id_contrato",
    right_index=True,
    how="left",
)

df["log_prazo_original"] = np.log1p(
    df["prazo_original_dias"]
)

df["log_qtd_parcelas_originais"] = np.log1p(
    df["qtd_parcelas_originais"]
)

print(
    "Contratos com informação de parcelas originais:",
    int(df["qtd_parcelas_originais"].notna().sum()),
    "de",
    len(df),
)

## 6. Indicadores de execução observados

Essas colunas ajudam a enriquecer o dataset, mas **não entram no modelo AFT atual**.

Para contratos concluídos, são calculadas até a conclusão.
Para contratos censurados, são calculadas até a data de corte.

Isso preserva uma visão coerente do que foi efetivamente observado.

In [ ]:
historico_rows = []

for _, row in df.iterrows():

    contrato = row["id_contrato"]
    cutoff = row["data_fim_observacao"]

    ap = aditivos_prazo[
        aditivos_prazo["idContrato"].eq(contrato)
        & (
            aditivos_prazo[
                "Data_assinatura_ta_prazo"
            ]
            <= cutoff
        )
    ]

    av = aditivos_valor[
        aditivos_valor["idContrato"].eq(contrato)
        & (
            aditivos_valor[
                "Data_assinatura_TA_valor"
            ]
            <= cutoff
        )
    ]

    pp = parcelas[
        parcelas["id_contrato"].eq(contrato)
        & parcelas["data_efetiva"].notna()
        & (
            parcelas["data_efetiva"]
            <= cutoff
        )
    ]

    ll = liberacoes[
        liberacoes["id_Contrato"].eq(contrato)
        & liberacoes["Data_liberação"].notna()
        & (
            liberacoes["Data_liberação"]
            <= cutoff
        )
    ]

    historico_rows.append({
        "id_contrato": contrato,
        "qtd_aditivos_prazo_observados": len(ap),
        "dias_aditados_observados": (
            pd.to_numeric(
                ap["Aditado_em_dias"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        ),
        "qtd_aditivos_valor_observados": len(av),
        "qtd_parcelas_concluidas_observadas": (
            pp["id_parcela"].nunique()
        ),
        "qtd_liberacoes_observadas": len(ll),
        "valor_liberado_observado": (
            pd.to_numeric(
                ll["Valor_total_liberado"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        ),
        "atraso_medio_liberacao_observado": (
            pd.to_numeric(
                ll["Atraso_considerado"],
                errors="coerce",
            ).mean()
            if len(ll)
            else np.nan
        ),
    })

historico = pd.DataFrame(historico_rows)

df = df.merge(
    historico,
    on="id_contrato",
    how="left",
)

df["pct_tempo_observado_sobre_prazo_atual"] = (
    df["duracao_observada_dias"]
    / df["prazo_vigente_observado_dias"]
)

display(
    df[
        [
            "id_contrato",
            "evento_conclusao",
            "duracao_observada_dias",
            "prazo_vigente_observado_dias",
            "qtd_aditivos_prazo_observados",
            "qtd_parcelas_concluidas_observadas",
            "qtd_liberacoes_observadas",
        ]
    ].head()
)

## 7. Exportação do dataset tratado

In [ ]:
DATASET_PATH = (
    OUTPUT_DIR
    / "dataset_survival_contratos.csv"
)

df.to_csv(
    DATASET_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Dataset salvo em:")
print(DATASET_PATH)
print()
print("Shape:", df.shape)

## 8. Dicionário resumido do dataset

As colunas de histórico são úteis para análise, mas ficam explicitamente marcadas como **não usadas no modelo atual** para evitar confusão.

In [ ]:
dicionario = pd.DataFrame([
    {
        "variavel": "duracao_observada_dias",
        "papel": "target_survival",
        "uso_modelo_atual": True,
        "descricao": "Dias entre OS e conclusão; para censurados, dias entre OS e a data de corte.",
    },
    {
        "variavel": "evento_conclusao",
        "papel": "target_survival",
        "uso_modelo_atual": True,
        "descricao": "1 se a conclusão foi observada; 0 se o contrato está censurado.",
    },
    {
        "variavel": "target_atraso_conhecido",
        "papel": "target_auxiliar",
        "uso_modelo_atual": False,
        "descricao": "1 atraso já conhecido; 0 concluído no prazo; NaN quando ainda não é possível saber.",
    },
    {
        "variavel": "prazo_original_dias",
        "papel": "feature_baseline",
        "uso_modelo_atual": True,
        "descricao": "Prazo original do contrato em dias.",
    },
    {
        "variavel": "classificacao_contrato",
        "papel": "feature_baseline",
        "uso_modelo_atual": True,
        "descricao": "Classificação da obra.",
    },
    {
        "variavel": "qtd_parcelas_originais",
        "papel": "feature_baseline",
        "uso_modelo_atual": True,
        "descricao": "Quantidade de parcelas do tipo Serviços Originais.",
    },
    {
        "variavel": "qtd_aditivos_prazo_observados",
        "papel": "historico_execucao",
        "uso_modelo_atual": False,
        "descricao": "Aditivos de prazo observados até conclusão/corte.",
    },
    {
        "variavel": "dias_aditados_observados",
        "papel": "historico_execucao",
        "uso_modelo_atual": False,
        "descricao": "Dias aditados observados até conclusão/corte.",
    },
    {
        "variavel": "qtd_aditivos_valor_observados",
        "papel": "historico_execucao",
        "uso_modelo_atual": False,
        "descricao": "Aditivos de valor observados até conclusão/corte.",
    },
    {
        "variavel": "qtd_parcelas_concluidas_observadas",
        "papel": "historico_execucao",
        "uso_modelo_atual": False,
        "descricao": "Parcelas concluídas observadas até conclusão/corte.",
    },
    {
        "variavel": "qtd_liberacoes_observadas",
        "papel": "historico_execucao",
        "uso_modelo_atual": False,
        "descricao": "Liberações observadas até conclusão/corte.",
    },
])

DICIONARIO_PATH = (
    OUTPUT_DIR
    / "dicionario_dataset_survival.csv"
)

dicionario.to_csv(
    DICIONARIO_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(dicionario)

# Parte 2 — Seleção do modelo

## Por que AFT Log-Normal?

Com apenas 64 contratos, 50 conclusões observadas e 14 censurados, um modelo de árvores de sobrevivência seria mais propenso a sobreajuste.

O modelo **Accelerated Failure Time (AFT) Log-Normal** é adequado porque:

- usa diretamente observações censuradas;
- é parcimonioso;
- estima uma distribuição de tempo até conclusão;
- permite obter uma mediana estimada de conclusão;
- permite calcular a probabilidade de ultrapassar uma data-limite.

A regularização L2 é usada para reduzir instabilidade.

In [ ]:
class LogNormalAFT:
    # Modelo AFT Log-Normal com censura à direita e penalização L2.
    def __init__(self, alpha=100.0):
        self.alpha = alpha

    def fit(self, X, tempo, evento):
        X = np.asarray(X, dtype=float)
        tempo = np.asarray(tempo, dtype=float)
        evento = np.asarray(evento, dtype=int)

        y = np.log(
            np.maximum(
                tempo,
                1e-6,
            )
        )

        # Inclui intercepto.
        X1 = np.column_stack([
            np.ones(len(X)),
            X,
        ])

        p = X1.shape[1]

        inicial = np.zeros(p + 1)
        inicial[0] = np.median(y)
        inicial[-1] = np.log(
            np.std(y) + 1e-3
        )

        def objetivo(theta):
            beta = theta[:p]
            log_sigma = theta[p]
            sigma = np.exp(log_sigma)

            z = (
                y - X1 @ beta
            ) / sigma

            # Para evento observado usamos densidade.
            # Para censura usamos a probabilidade de sobreviver além
            # do tempo observado.
            loglik = np.where(
                evento == 1,
                norm.logpdf(z)
                - log_sigma
                - y,
                norm.logsf(z),
            )

            # Penaliza coeficientes, exceto o intercepto.
            penalizacao = (
                0.5
                * self.alpha
                * np.sum(
                    beta[1:] ** 2
                )
            )

            return (
                -np.sum(loglik)
                + penalizacao
            )

        self.resultado_ = minimize(
            objetivo,
            inicial,
            method="L-BFGS-B",
            bounds=(
                [(None, None)] * p
                + [(-5, 3)]
            ),
        )

        self.beta_ = (
            self.resultado_.x[:p]
        )

        self.sigma_ = np.exp(
            self.resultado_.x[p]
        )

        return self

    def mu(self, X):
        X = np.asarray(
            X,
            dtype=float,
        )

        X1 = np.column_stack([
            np.ones(len(X)),
            X,
        ])

        return X1 @ self.beta_

    def mediana_tempo(self, X):
        return np.exp(
            self.mu(X)
        )

    def sobrevivencia(self, X, tempo):
        mu = self.mu(X)

        tempo = np.maximum(
            np.asarray(
                tempo,
                dtype=float,
            ),
            1e-6,
        )

        z = (
            np.log(tempo)
            - mu
        ) / self.sigma_

        return norm.sf(z)

    def risco_condicional_prazo(
        self,
        X,
        tempo_decorrido,
        prazo_limite,
    ):
        # P(T > prazo | T > tempo_decorrido)
        tempo_decorrido = np.asarray(
            tempo_decorrido,
            dtype=float,
        )

        prazo_limite = np.asarray(
            prazo_limite,
            dtype=float,
        )

        S_decorrido = self.sobrevivencia(
            X,
            tempo_decorrido,
        )

        S_prazo = self.sobrevivencia(
            X,
            prazo_limite,
        )

        risco = (
            S_prazo
            / np.maximum(
                S_decorrido,
                1e-12,
            )
        )

        # Se o contrato já passou do prazo e ainda não terminou,
        # o atraso já é conhecido.
        risco = np.where(
            prazo_limite
            <= tempo_decorrido,
            1.0,
            risco,
        )

        return np.clip(
            risco,
            0,
            1,
        )

    def mediana_condicional(
        self,
        X,
        tempo_decorrido,
    ):
        # Mediana de T condicionada a T > tempo_decorrido.
        mu = self.mu(X)

        tempo_decorrido = np.maximum(
            np.asarray(
                tempo_decorrido,
                dtype=float,
            ),
            1e-6,
        )

        S_decorrido = self.sobrevivencia(
            X,
            tempo_decorrido,
        )

        alvo_cdf = (
            1
            - 0.5
            * S_decorrido
        )

        z = norm.ppf(
            np.clip(
                alvo_cdf,
                1e-8,
                1 - 1e-8,
            )
        )

        return np.exp(
            mu
            + self.sigma_ * z
        )

## 9. Pré-processamento e métrica de sobrevivência

In [ ]:
def make_preprocessor(
    numeric_features,
    categorical_features,
):
    transformers = []

    if numeric_features:
        transformers.append((
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]),
            numeric_features,
        ))

    if categorical_features:
        transformers.append((
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent",
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        drop="first",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical_features,
        ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


def harrell_c_index(
    tempo,
    evento,
    risco,
):
    # Risco maior = conclusão esperada mais cedo.
    tempo = np.asarray(
        tempo,
        dtype=float,
    )

    evento = np.asarray(
        evento,
        dtype=int,
    )

    risco = np.asarray(
        risco,
        dtype=float,
    )

    concordantes = 0.0
    empates = 0.0
    comparaveis = 0.0

    for i in range(len(tempo)):
        for j in range(i + 1, len(tempo)):

            if (
                tempo[i] < tempo[j]
                and evento[i] == 1
            ):
                comparaveis += 1

                if risco[i] > risco[j]:
                    concordantes += 1
                elif risco[i] == risco[j]:
                    empates += 1

            elif (
                tempo[j] < tempo[i]
                and evento[j] == 1
            ):
                comparaveis += 1

                if risco[j] > risco[i]:
                    concordantes += 1
                elif risco[i] == risco[j]:
                    empates += 1

    if comparaveis == 0:
        return np.nan

    return (
        concordantes
        + 0.5 * empates
    ) / comparaveis

## 10. Validação temporal

Os contratos são ordenados pela data da OS.

Usamos janelas expansivas:
- 30 contratos de treino + próximos 8 de validação;
- 35 + 8;
- 40 + 8;
- 45 + 8;
- 50 + 8.

O C-index é a métrica principal porque lida com tempos censurados.

Também calculamos Brier para `target_atraso_conhecido` como diagnóstico secundário. Esse Brier só usa contratos cujo atraso já é observável.

In [ ]:
model_data = (
    df
    .sort_values(
        [
            "data_os",
            "id_contrato",
        ]
    )
    .reset_index(drop=True)
)

FOLD_STARTS = [
    30,
    35,
    40,
    45,
    50,
]

folds = []

for n_train in FOLD_STARTS:

    train = (
        model_data
        .iloc[:n_train]
        .copy()
    )

    val = (
        model_data
        .iloc[
            n_train:n_train + 8
        ]
        .copy()
    )

    if len(val) < 5:
        continue

    folds.append({
        "n_train": n_train,
        "train": train,
        "val": val,
    })

display(
    pd.DataFrame([
        {
            "n_train": f["n_train"],
            "n_validacao": len(f["val"]),
            "eventos_validacao": int(
                f["val"][
                    "evento_conclusao"
                ].sum()
            ),
            "inicio_validacao": (
                f["val"]["data_os"].min()
            ),
            "fim_validacao": (
                f["val"]["data_os"].max()
            ),
        }
        for f in folds
    ])
)

## 11. Candidatos

Mantemos poucos candidatos de propósito.

Quanto menor a base, maior o risco de escolher um modelo complexo que apenas memoriza os 64 contratos.

In [ ]:
CANDIDATOS = {
    "AFT_prazo": {
        "numericas": [
            "log_prazo_original",
        ],
        "categoricas": [],
    },

    "AFT_prazo_classificacao": {
        "numericas": [
            "log_prazo_original",
        ],
        "categoricas": [
            "classificacao_contrato",
        ],
    },

    "AFT_prazo_classificacao_parcelas": {
        "numericas": [
            "log_prazo_original",
            "log_qtd_parcelas_originais",
        ],
        "categoricas": [
            "classificacao_contrato",
        ],
    },
}

ALPHAS = [
    10.0,
    100.0,
]

print("Candidatos:")
for nome, cfg in CANDIDATOS.items():
    print(
        nome,
        "->",
        cfg["numericas"]
        + cfg["categoricas"],
    )

## 12. Execução da validação temporal

In [ ]:
cv_rows = []

for modelo_nome, config in CANDIDATOS.items():

    num = config["numericas"]
    cat = config["categoricas"]
    features = num + cat

    for alpha in ALPHAS:

        for fold in folds:

            train = fold["train"]
            val = fold["val"]

            prep = make_preprocessor(
                num,
                cat,
            )

            X_train = prep.fit_transform(
                train[features]
            )

            X_val = prep.transform(
                val[features]
            )

            model = LogNormalAFT(
                alpha=alpha
            )

            model.fit(
                X_train,
                train[
                    "duracao_observada_dias"
                ],
                train[
                    "evento_conclusao"
                ],
            )

            # Para o C-index:
            # menor tempo estimado = maior risco de conclusão mais cedo.
            risco_conclusao = (
                -model.mu(X_val)
            )

            cindex = harrell_c_index(
                val[
                    "duracao_observada_dias"
                ],
                val[
                    "evento_conclusao"
                ],
                risco_conclusao,
            )

            # Diagnóstico secundário de atraso.
            # Probabilidade de não concluir até o prazo vigente observado.
            prob_atraso = model.sobrevivencia(
                X_val,
                val[
                    "prazo_vigente_observado_dias"
                ].to_numpy(),
            )

            mask = (
                val[
                    "target_atraso_conhecido"
                ].notna()
            )

            if mask.sum():

                brier = brier_score_loss(
                    val.loc[
                        mask,
                        "target_atraso_conhecido",
                    ],
                    prob_atraso[mask.to_numpy()],
                )

            else:
                brier = np.nan

            cv_rows.append({
                "modelo": modelo_nome,
                "alpha": alpha,
                "n_train": fold["n_train"],
                "n_validacao": len(val),
                "eventos_validacao": int(
                    val[
                        "evento_conclusao"
                    ].sum()
                ),
                "n_atraso_conhecido": int(
                    mask.sum()
                ),
                "cindex": cindex,
                "brier_atraso_conhecido": brier,
                "otimizacao_ok": (
                    model.resultado_.success
                ),
            })


cv = pd.DataFrame(
    cv_rows
)

display(
    cv.sort_values(
        [
            "n_train",
            "cindex",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

## 13. Resumo e seleção automática

In [ ]:
resumo = (
    cv
    .groupby(
        [
            "modelo",
            "alpha",
        ],
        as_index=False,
    )
    .agg(
        cindex_medio=(
            "cindex",
            "mean",
        ),
        cindex_dp=(
            "cindex",
            "std",
        ),
        brier_medio=(
            "brier_atraso_conhecido",
            "mean",
        ),
        n_validacoes=(
            "cindex",
            "count",
        ),
        n_brier=(
            "n_atraso_conhecido",
            "sum",
        ),
    )
)

# Critério principal: maior C-index médio.
# Em caso de desempenho muito parecido, menor Brier ajuda a desempatar.
resumo = resumo.sort_values(
    [
        "cindex_medio",
        "brier_medio",
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(drop=True)

display(resumo)

BEST_MODEL_NAME = resumo.iloc[0]["modelo"]
BEST_ALPHA = float(
    resumo.iloc[0]["alpha"]
)

print(
    "Modelo selecionado:",
    BEST_MODEL_NAME,
)

print(
    "Alpha:",
    BEST_ALPHA,
)

### Benchmark simples

Para não interpretar o C-index isoladamente, comparamos o modelo selecionado com dois benchmarks:

- **prazo original como previsão de duração**: quanto maior o prazo, maior o tempo esperado até conclusão;
- **prevalência histórica de atraso**: probabilidade constante igual à taxa observada de atraso no treino.

Se o modelo sofisticado ficar muito próximo desses benchmarks, o ganho preditivo ainda é pequeno.

In [ ]:
baseline_rows = []

for fold in folds:

    train = fold["train"]
    val = fold["val"]

    # Benchmark de ranking de duração:
    # prazo maior -> conclusão esperada mais tarde.
    cindex_prazo = harrell_c_index(
        val["duracao_observada_dias"],
        val["evento_conclusao"],
        -val["prazo_original_dias"],
    )

    # Benchmark probabilístico de atraso:
    # usa apenas contratos do treino cujo atraso já é conhecido.
    train_known = train[
        train["target_atraso_conhecido"].notna()
    ]

    prevalencia = (
        train_known[
            "target_atraso_conhecido"
        ].mean()
    )

    val_known = val[
        val["target_atraso_conhecido"].notna()
    ]

    if len(val_known):
        brier_prevalencia = brier_score_loss(
            val_known["target_atraso_conhecido"],
            np.full(
                len(val_known),
                prevalencia,
            ),
        )
    else:
        brier_prevalencia = np.nan

    baseline_rows.append({
        "n_train": fold["n_train"],
        "cindex_baseline_prazo": cindex_prazo,
        "prevalencia_atraso_treino": prevalencia,
        "brier_baseline_prevalencia": brier_prevalencia,
        "n_atraso_conhecido_validacao": len(val_known),
    })

baseline_cv = pd.DataFrame(
    baseline_rows
)

display(baseline_cv)

print(
    "C-index médio do baseline de prazo:",
    round(
        baseline_cv[
            "cindex_baseline_prazo"
        ].mean(),
        4,
    ),
)

print(
    "Brier médio do baseline de prevalência:",
    round(
        baseline_cv[
            "brier_baseline_prevalencia"
        ].mean(),
        4,
    ),
)

print(
    "C-index médio do melhor AFT:",
    round(
        resumo.iloc[0][
            "cindex_medio"
        ],
        4,
    ),
)

print(
    "Brier médio do melhor AFT:",
    round(
        resumo.iloc[0][
            "brier_medio"
        ],
        4,
    ),
)

**Leitura importante:** o C-index mede ordenação do tempo até conclusão, não calibração de atraso. Se o AFT superar pouco o baseline de prazo, ele é útil como estrutura para censura, mas ainda não demonstra grande ganho preditivo. Se o Brier for pior que a prevalência, as probabilidades de atraso devem ser tratadas como exploratórias e não como probabilidades calibradas de produção.

## 14. Comparação visual

Em C-index:
- 0,50 ≈ ranking aleatório;
- quanto mais próximo de 1, melhor a ordenação do tempo até conclusão.

Como a base ainda é pequena, diferenças de poucos centésimos devem ser interpretadas com cautela.

In [ ]:
plot_df = (
    resumo
    .copy()
)

plot_df["rotulo"] = (
    plot_df["modelo"]
    + " | alpha="
    + plot_df["alpha"].astype(str)
)

plt.figure(
    figsize=(10, 5)
)

plt.barh(
    plot_df["rotulo"],
    plot_df["cindex_medio"],
)

plt.axvline(
    0.5,
    linewidth=1,
)

plt.xlabel(
    "Harrell C-index médio"
)

plt.title(
    "Validação temporal dos modelos AFT"
)

plt.tight_layout()
plt.show()

## 15. Ajuste do modelo final com os 64 contratos

Agora usamos os 64 contratos — incluindo os censurados — para estimar o modelo final.

In [ ]:
best_config = CANDIDATOS[
    BEST_MODEL_NAME
]

BEST_NUMERIC = (
    best_config["numericas"]
)

BEST_CATEGORICAL = (
    best_config["categoricas"]
)

BEST_FEATURES = (
    BEST_NUMERIC
    + BEST_CATEGORICAL
)

final_preprocessor = (
    make_preprocessor(
        BEST_NUMERIC,
        BEST_CATEGORICAL,
    )
)

X_all = (
    final_preprocessor
    .fit_transform(
        df[BEST_FEATURES]
    )
)

final_model = LogNormalAFT(
    alpha=BEST_ALPHA
)

final_model.fit(
    X_all,
    df[
        "duracao_observada_dias"
    ],
    df[
        "evento_conclusao"
    ],
)

print(
    "Otimização convergiu:",
    final_model.resultado_.success,
)

print(
    "Sigma estimado:",
    round(
        final_model.sigma_,
        4,
    ),
)

print(
    "Features finais:",
    BEST_FEATURES,
)

## 16. Risco atual dos contratos ainda não concluídos

Para um contrato em andamento, já sabemos que ele sobreviveu sem conclusão até a data de corte.

Por isso calculamos:

`P(concluir depois do prazo | ainda não concluiu até hoje)`

Se o contrato já ultrapassou o prazo vigente e ainda não concluiu, essa probabilidade é 1 por definição.

In [ ]:
ongoing = df[
    df["evento_conclusao"].eq(0)
].copy()

X_ongoing = (
    final_preprocessor
    .transform(
        ongoing[BEST_FEATURES]
    )
)

tempo_decorrido = (
    ongoing[
        "duracao_observada_dias"
    ].to_numpy()
)

prazo_limite = (
    ongoing[
        "prazo_vigente_observado_dias"
    ].to_numpy()
)

ongoing[
    "prob_atraso_condicional"
] = (
    final_model
    .risco_condicional_prazo(
        X_ongoing,
        tempo_decorrido,
        prazo_limite,
    )
)

ongoing[
    "duracao_mediana_condicional"
] = (
    final_model
    .mediana_condicional(
        X_ongoing,
        tempo_decorrido,
    )
)

ongoing[
    "data_conclusao_mediana_estimada"
] = (
    ongoing["data_os"]
    + pd.to_timedelta(
        np.round(
            ongoing[
                "duracao_mediana_condicional"
            ]
        ).astype(int),
        unit="D",
    )
)

ongoing[
    "dias_atraso_mediana_vs_prazo_atual"
] = np.maximum(
    0,
    ongoing[
        "duracao_mediana_condicional"
    ]
    - ongoing[
        "prazo_vigente_observado_dias"
    ],
)

resultado_ongoing = ongoing[
    [
        "id_contrato",
        "classificacao_contrato",
        "status_final_controle",
        "data_os",
        "data_termino_vigente_observada",
        "duracao_observada_dias",
        "prazo_vigente_observado_dias",
        "prob_atraso_condicional",
        "data_conclusao_mediana_estimada",
        "dias_atraso_mediana_vs_prazo_atual",
    ]
].sort_values(
    "prob_atraso_condicional",
    ascending=False,
)

display(
    resultado_ongoing.round(3)
)

## 17. Exportação dos resultados

O arquivo principal para modelagem é `dataset_survival_contratos.csv`.

As previsões dos contratos em andamento devem ser tratadas como **estimativas exploratórias**, porque a base ainda é pequena e algumas classificações novas possuem poucos ou nenhum evento concluído.

In [ ]:
CV_PATH = (
    OUTPUT_DIR
    / "validacao_temporal_aft.csv"
)

RESUMO_PATH = (
    OUTPUT_DIR
    / "resumo_modelos_aft.csv"
)

ONGOING_PATH = (
    OUTPUT_DIR
    / "previsoes_contratos_em_andamento.csv"
)

BASELINE_PATH = (
    OUTPUT_DIR
    / "baseline_validacao_temporal.csv"
)

cv.to_csv(
    CV_PATH,
    index=False,
    encoding="utf-8-sig",
)

resumo.to_csv(
    RESUMO_PATH,
    index=False,
    encoding="utf-8-sig",
)

resultado_ongoing.to_csv(
    ONGOING_PATH,
    index=False,
    encoding="utf-8-sig",
)

baseline_cv.to_csv(
    BASELINE_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Arquivos gerados:")
print("-", DATASET_PATH)
print("-", DICIONARIO_PATH)
print("-", CV_PATH)
print("-", RESUMO_PATH)
print("-", ONGOING_PATH)

# Conclusão metodológica

Para a realidade atual dos dados, a recomendação é:

**Target principal**
- `duracao_observada_dias`
- `evento_conclusao`

**Modelo**
- AFT Log-Normal regularizado, escolhido por validação temporal para o target de sobrevivência.
- O benchmark simples é mantido no notebook para verificar se o ganho do AFT é realmente material.

**Target de atraso**
- `target_atraso_conhecido` continua útil para auditoria e avaliação, mas não deve ser o único target principal porque 12 contratos ainda não têm desfecho de atraso observável.

**Próximo avanço**
Quando houver mais contratos concluídos, podemos construir um modelo temporal com features de execução por snapshots ou landmarks. Por enquanto, o modelo AFT basal é mais defensável porque usa os 64 contratos sem transformar contratos em andamento em falsos negativos.